# Capítulo 2 — Probabilidade e Distribuições

Notebook com o **código** deste capítulo, para o Google Colab. Cada trecho vem precedido de uma explicação curta; o texto completo está no site do livro.

Rode a célula de **setup** abaixo primeiro (uma vez), depois as demais em ordem.

In [ ]:
# Setup (rode uma vez).
!curl -sO https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/formato.py   # baixa o ajudante de formatação do livro

## 2.1 — O que é Probabilidade

Simula 10.000 lançamentos de uma moeda justa e acompanha a proporção acumulada de caras em diferentes marcos (10, 100, 1.000, 10.000). É a definição frequentista de probabilidade posta em prática: a proporção observada oscila bastante no começo e só se aproxima de 0,5 com muitas repetições.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)
rng = np.random.default_rng(42)
lancamentos = rng.integers(0, 2, 10000)          # 0 ou 1, moeda justa
proporcao = np.cumsum(lancamentos) / np.arange(1, 10001)

for n in [10, 100, 1000, 10000]:
    print(f"Após {n:>5} lançamentos: proporção de caras = {num(proporcao[n-1], 3)}")

Plota a proporção acumulada de caras contra o número de lançamentos, em escala logarítmica para dar visibilidade tanto às oscilações iniciais quanto à estabilização final. A curva mostra visualmente a convergência para 0,5 — o comportamento de longo prazo que a visão frequentista de probabilidade descreve.

In [ ]:
fig, ax = plt.subplots()
ax.plot(np.arange(1, 10001), proporcao, color="#2c3e50", linewidth=1)
ax.axhline(0.5, color="#c0392b", linestyle="--", linewidth=2)
ax.set_xscale("log")
ax.set_xlabel("Número de lançamentos (escala log)")
ax.set_ylabel("Proporção de caras")
plt.tight_layout()
plt.show()

## 2.2 — Regras de Probabilidade

Aplica a regra do complementar: se 2% das requisições falham, 98% não falham. Os dois números são a mesma informação vista de dois lados, e essa regra simples resolve em uma linha perguntas que, calculadas de frente, exigiriam somar vários casos.

In [ ]:
from formato import num

p_falha = 0.02
p_sucesso = 1 - p_falha
print(f"P(requisição falhar)     : {num(p_falha, 2)}")
print(f"P(requisição não falhar) : {num(p_sucesso, 2)}")

Usa a regra da multiplicação (independência) para calcular a chance de dois serviços independentes estarem ambos no ar, e o complementar para obter a chance de pelo menos um cair. O resultado mostra que "pelo menos um fora" é quase o dobro da falha de um serviço isolado, porque a queda de qualquer um dos dois já conta.

In [ ]:
from formato import num
p = 0.99
print(f"Ambos os serviços no ar:      {num(p**2, 4)}")
print(f"Pelo menos um fora do ar:     {num(1 - p**2, 4)}")

Constrói a distribuição de probabilidade completa do número de caras em dois lançamentos, combinando a regra da multiplicação (lançamentos independentes) com a da adição (duas formas mutuamente exclusivas de dar exatamente uma cara). O resultado — 0,25, 0,50, 0,25 — soma 1, como exige a certeza de que algum desses valores vai ocorrer.

In [ ]:
from formato import num

p_caras = {
    0: 0.5**2,           # coroa-coroa
    1: 2 * 0.5**2,       # cara-coroa OU coroa-cara
    2: 0.5**2,           # cara-cara
}
for k, prob in p_caras.items():
    print(f"P(X = {k}) = {num(prob, 2)}")

## 2.3 — Distribuição Binomial

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

Desenha o gráfico de barras da distribuição binomial com n=20 e p=0,1 — barras, não uma curva, porque o número de sucessos só assume valores inteiros. O pico fica perto de np=2, exatamente onde a média da binomial coloca o centro.

In [ ]:
k = np.arange(0, 11)
fig, ax = plt.subplots()
ax.bar(k, stats.binom.pmf(k, 20, 0.1), color="#b0c4d8", edgecolor="white")
ax.set_xlabel("Número de sucessos")
ax.set_ylabel("Probabilidade")
plt.tight_layout()
plt.show()

Compara a probabilidade de **exatamente** 2 sucessos (`pmf`) com a de **até** 2 sucessos (`cdf`) em 5 tentativas com p=0,1. A distância entre os dois números — cerca de 7% contra mais de 99% — é a assinatura de eventos raros: poucos sucessos concentram quase toda a massa lá embaixo.

In [ ]:
print(f"P(exatamente 2 sucessos em 5, p=0,1): {num(stats.binom.pmf(2, 5, 0.1), 4)}")
print(f"P(até 2 sucessos em 5, p=0,1):        {num(stats.binom.cdf(2, 5, 0.1), 4)}")

## 2.4 — Distribuição de Poisson e Relacionadas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

Desenha o gráfico de barras da distribuição de Poisson com taxa média λ=2. Como a binomial, é discreta — conta eventos inteiros —, e o pico fica em torno da própria taxa λ.

In [ ]:
k = np.arange(0, 11)
fig, ax = plt.subplots()
ax.bar(k, stats.poisson.pmf(k, 2), color="#b0c4d8", edgecolor="white")
ax.set_xlabel("Número de eventos no intervalo")
ax.set_ylabel("Probabilidade")
plt.tight_layout()
plt.show()

Calcula, para uma Poisson de taxa média 2, a chance de nenhum evento ocorrer e a de 5 ou mais ocorrerem no intervalo. Mesmo com taxa 2, ainda há mais de 13% de chance de o intervalo passar em branco, enquanto ver 5 ou mais é raro (cerca de 5%).

In [ ]:
print(f"Poisson(λ=2): P(nenhum evento) = {num(stats.poisson.pmf(0, 2), 3)}")
print(f"Poisson(λ=2): P(5 ou mais)     = {num(stats.poisson.sf(4, 2), 3)}")

Calcula o tempo médio de espera entre eventos — o inverso da taxa — usando a distribuição exponencial, a irmã contínua da Poisson que descreve **quanto tempo** até o próximo evento em vez de **quantos** eventos ocorrem.

In [ ]:
print(f"Se ocorrem 2 eventos por hora, o tempo médio entre eventos é {num(1 / 2 * 60, 0)} minutos.")

## 2.5 — Distribuição Normal

Carrega as rendas de solicitantes de empréstimo que servirão de exemplo real de dado não-normal nas células seguintes (QQ-plot e assimetria).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)
renda = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/loans_income.csv").rename(columns={"x": "Renda"})["Renda"]

Plota a densidade normal-padrão com marcações em ±1, ±2 e ±3 desvios, ilustrando a regra 68–95–99,7: cerca de 68% da massa fica a menos de um desvio da média, 95% a menos de dois, 99,7% a menos de três.

In [ ]:
x = np.linspace(-4, 4, 200)
fig, ax = plt.subplots()
ax.plot(x, stats.norm.pdf(x), color="#2c3e50", linewidth=2)
for k, cor in [(1, "#27ae60"), (2, "#e67e22"), (3, "#c0392b")]:
    ax.axvline(k, color=cor, linestyle="--", alpha=0.6)
    ax.axvline(-k, color=cor, linestyle="--", alpha=0.6)
ax.set_xlabel("Desvios em relação à média")
ax.set_ylabel("Densidade")
plt.tight_layout()
plt.show()

Constrói o QQ-plot da renda contra a distribuição normal: se a renda fosse normal, os pontos cairiam sobre a reta. O corpo segue a reta, mas a ponta direita curva para cima — a assinatura da cauda longa à direita da renda.

In [ ]:
fig, ax = plt.subplots()
stats.probplot(renda, dist="norm", plot=ax)
ax.set_title("")
ax.set_xlabel("Quantis teóricos (normal)")
ax.set_ylabel("Quantis da renda")
plt.tight_layout()
plt.show()

Calcula a assimetria da renda. O valor positivo confirma o que o QQ-plot já mostrava: a renda tem cauda longa à direita, bem diferente da assimetria zero de uma normal.

In [ ]:
print(f"Assimetria da renda: {num(stats.skew(renda), 2)}")

## 2.6 — Distribuições de Cauda Longa

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

Gera uma amostra de uma distribuição t com 3 graus de liberdade — genuinamente de cauda gorda — e mede sua curtose contra o valor zero de uma normal. A curtose alta confirma que essa amostra tem caudas muito mais pesadas que a normal.

In [ ]:
amostra = stats.t.rvs(df=3, size=1000, random_state=42)
print(f"Curtose da amostra:      {num(stats.kurtosis(amostra), 1)}")
print(f"Curtose de uma normal:   0")

Constrói o QQ-plot dessa amostra de cauda gorda contra a normal. A curva em S — subindo à direita e descendo à esquerda — é a marca da cauda gorda: por ser simétrica, ela infla os dois extremos ao mesmo tempo, diferente da curva de um lado só da renda assimétrica.

In [ ]:
fig, ax = plt.subplots()
stats.probplot(amostra, dist="norm", plot=ax)
ax.set_title("")
ax.set_xlabel("Quantis teóricos (normal)")
ax.set_ylabel("Quantis da amostra")
plt.tight_layout()
plt.show()

## 2.7 — Distribuição t de Student

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

Plota a densidade normal ao lado da t com 1 e com 5 graus de liberdade. A t(1) é a mais achatada no centro e mais gorda nas caudas; a t(5) já se aproxima bem mais da normal — conforme os graus de liberdade crescem, a t converge para a normal.

In [ ]:
x = np.linspace(-4, 4, 200)
fig, ax = plt.subplots()
ax.plot(x, stats.norm.pdf(x), color="#2c3e50", linewidth=2, label="Normal")
for gl, cor in [(1, "#c0392b"), (5, "#e67e22")]:
    ax.plot(x, stats.t.pdf(x, gl), color=cor, linewidth=2, linestyle="--", label=f"t({gl})")
ax.set_xlabel("Valor")
ax.set_ylabel("Densidade")
ax.legend()
plt.tight_layout()
plt.show()

Tabela a probabilidade de um valor ultrapassar ±2 para a t com diferentes graus de liberdade e para a normal. Com poucos graus de liberdade essa cauda é enorme (quase um terço da massa com 1 grau); com 30, o valor já fica colado ao da normal — a base da "regra do 30".

In [ ]:
print(f"{'distribuição':>14}  {'P(|X| > 2)':>10}")
for gl in [1, 5, 30]:
    print(f"{'t(' + str(gl) + ')':>14}  {num(2 * stats.t.sf(2, gl), 3):>10}")
print(f"{'normal':>14}  {num(2 * stats.norm.sf(2), 3):>10}")

## 2.8 — Distribuição Qui-Quadrado

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

Plota a densidade qui-quadrado para diferentes graus de liberdade. É sempre assimétrica à direita e só assume valores positivos — soma de quadrados nunca é negativa —, e o pico se afasta do zero conforme os graus de liberdade crescem.

In [ ]:
x = np.linspace(0, 20, 200)
fig, ax = plt.subplots()
for gl, cor in [(2, "#27ae60"), (5, "#e67e22"), (10, "#c0392b")]:
    ax.plot(x, stats.chi2.pdf(x, gl), color=cor, linewidth=2, label=f"k = {gl}")
ax.set_xlabel("Valor")
ax.set_ylabel("Densidade")
ax.legend()
plt.tight_layout()
plt.show()

Calcula a média de χ²(5), que é igual aos próprios graus de liberdade, e o valor crítico de 5% — o ponto acima do qual fica só 5% da distribuição. É contra esse valor crítico que um teste qui-quadrado compara sua estatística para decidir se um desvio é grande demais para ser acaso.

In [ ]:
print(f"Média de χ²(5):            {num(stats.chi2.mean(5), 0)}  (= graus de liberdade)")
print(f"Valor crítico 5% de χ²(5): {num(stats.chi2.ppf(0.95, 5), 2)}")

## 2.9 — Distribuição F

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

Plota a densidade F para dois pares de graus de liberdade (numerador e denominador). Como é uma razão de variâncias, é sempre positiva e assimétrica à direita, ficando perto de 1 quando as duas variâncias comparadas são iguais.

In [ ]:
x = np.linspace(0, 5, 200)
fig, ax = plt.subplots()
for (g1, g2), cor in [((5, 10), "#27ae60"), ((10, 30), "#e67e22")]:
    ax.plot(x, stats.f.pdf(x, g1, g2), color=cor, linewidth=2, label=f"F({g1}, {g2})")
ax.set_xlabel("Valor")
ax.set_ylabel("Densidade")
ax.legend()
plt.tight_layout()
plt.show()

Calcula o valor crítico de 5% da distribuição F(5, 10) — o limiar acima do qual uma razão de variâncias tem menos de 5% de chance de ocorrer só por acaso. É essa lógica que sustenta a ANOVA, comparando a variação entre grupos com a variação dentro deles.

In [ ]:
print(f"Valor crítico 5% de F(5, 10): {num(stats.f.ppf(0.95, 5, 10), 2)}")